# Steps 4–5: Results Routing & Clinical Follow-Up Pathways

This notebook covers what happens **after a screening result is returned** — the clinical decision-making layer that determines each patient's next step.

## What this notebook covers

**Step 4 — Result Routing**

A screening result doesn't directly produce an outcome — it triggers a routing decision. Depending on the result category, the patient is sent to:
- **Routine surveillance** (NORMAL, HPV_NEGATIVE): return to the regular screening interval, no immediate action.
- **Colposcopy referral** (ASCUS, LSIL, ASC-H, HSIL): an office procedure to visually inspect the cervix and take biopsies.
- **1-year repeat cytology** (some HPV_POSITIVE patients): low-risk management path per ASCCP guidelines.
- **LTFU** (Loss-to-Follow-Up): patient does not follow through on the referral. This is a key care gap in real-world settings and is explicitly modelled here.

Lung routing follows Lung-RADS: low/benign results → repeat LDCT at the appropriate interval; suspicious results (RADS 4A/4B/4X) → biopsy pathway.

**Step 5 — Follow-Up Execution**

For patients who complete a referral:
- **Colposcopy** draws a CIN grade (NORMAL, CIN1, CIN2, CIN3) from a probability table conditioned on the triggering result. Higher-grade triggers are more likely to produce high-grade CIN.
- **Treatment**: CIN1/NORMAL → surveillance (watchful waiting); CIN2/CIN3 → LEEP excision.
- **Lung biopsy**: five sequential steps, each with its own LTFU probability.

All probability values are **PLACEHOLDERS** — replace with NYP / ASCCP risk-table values before using for planning.

---
All logic lives in `followup.py`. This notebook demonstrates and validates each component individually, then runs visualisations of the resulting distributions.

In [ ]:
import sys, random
sys.path.insert(0, '../src')   # add src/ so we can import local .py modules

import config as cfg
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from patient import Patient
from population import sample_patient          # generates synthetic patients
from metrics import initialize_metrics, record_exit, print_summary
from followup import (
    route_cervical_result,    # decides next step after a cervical result
    run_colposcopy,           # draws CIN grade from colposcopy
    run_treatment,            # assigns treatment based on CIN grade (with LTFU check)
    run_cervical_followup,    # full cervical pipeline: routing → colposcopy → treatment
    run_lung_followup,        # full lung pipeline: RADS routing → biopsy → treatment
)
from screening import run_screening_step, get_eligible_screenings

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

## Cervical Result Routing

Each cervical result category maps to a distinct clinical pathway. Understanding this mapping is essential because the routing decision determines:
1. **Whether a colposcopy is needed** — a downstream resource-intensive procedure.
2. **Where LTFU risk is introduced** — patients with abnormal results can drop out before attending colposcopy.
3. **What the simulation counts as a care gap** — unmet colposcopy referrals are the primary LTFU metric for cervical cancer.

`route_cervical_result` makes a stochastic LTFU draw before every colposcopy referral. Running 200 trials per category shows the *distribution* of routing outcomes rather than a single deterministic path. The LTFU probability is set in `config.LTFU_PROBS['post_abnormal_to_colposcopy']`.

In [ ]:
# Valid cervical result categories by test type:
#   Cytology (Pap smear): NORMAL, ASCUS, LSIL, ASC-H, HSIL
#   HPV-alone test:       HPV_NEGATIVE, HPV_POSITIVE
# Each maps to a different clinical pathway in followup.py
results_to_test = ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_NEGATIVE', 'HPV_POSITIVE']

# Run 200 trials per category: route_cervical_result() internally draws a random number
# to decide whether the patient is lost (LTFU) before colposcopy. A single run would
# just return one outcome; 200 trials shows the full routing probability distribution.
N_TRIALS = 200

print(f'{"Result":<15}  Routing distribution ({N_TRIALS} trials each)')
print('-' * 75)
for result in results_to_test:
    routes = []
    for _ in range(N_TRIALS):
        # Create a fresh patient each trial — route_cervical_result() writes to p.active
        # and p.exit_reason if the patient is lost, so we must not reuse the same object
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, result
        routes.append(route_cervical_result(p, 0))

    # Count how many times each routing outcome appeared across the 200 trials
    cnt     = Counter(routes)
    summary = '  '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {result:<13}  {summary}')

## Colposcopy CIN Grade Distribution

Colposcopy is not just a confirmatory test — it produces a **CIN grade** that directly determines treatment. The grade distribution depends critically on what triggered the colposcopy:
- A patient referred for ASCUS (mildest abnormality) is much more likely to have NORMAL or CIN1 on colposcopy.
- A patient referred for HSIL (high-grade abnormality) is far more likely to have CIN2 or CIN3, which require excisional treatment.

This gradient is captured in `config.COLPOSCOPY_RESULT_PROBS`, where each entry is a probability table keyed by the triggering result (e.g. `"from_HSIL"`). Verifying that higher-grade triggers produce higher-grade CIN distributions confirms the config is internally consistent and clinically plausible.

We run 1,000 trials per trigger to get stable frequency estimates.

In [ ]:
# Only abnormal results trigger colposcopy — NORMAL and HPV_NEGATIVE go to surveillance
triggering_results = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
N = 1_000

for trigger in triggering_results:
    cin_counts = Counter()
    for _ in range(N):
        # Fresh patient per trial: run_colposcopy() sets p.colposcopy_result and p.current_stage.
        # If we reused the same patient, those fields would already be set from the previous trial
        # and could interfere with the draw logic.
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, trigger

        # run_colposcopy() looks up the config table keyed on "from_{trigger}"
        # (e.g. "from_HSIL") and does a weighted random draw to get a CIN grade.
        cin_counts[run_colposcopy(p, 0)] += 1

    # Look up the config table for this trigger to get the expected probabilities
    expected = cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{trigger}', {})
    print(f'\nColposcopy from {trigger} (n={N:,}):')
    for grade in ['NORMAL', 'CIN1', 'CIN2', 'CIN3']:
        obs = cin_counts.get(grade, 0) / N * 100
        exp = expected.get(grade, 0) * 100
        print(f'  {grade:<8} observed={obs:5.1f}%  expected={exp:5.1f}%')

## Full Cervical Follow-Up — End-to-End Patient Traces

This section runs individual patients through the **complete cervical pathway** from screening result through to their final disposition. This is the most direct way to verify that the clinical chain is logically sound.

Each patient trace shows every logged event in order: screening result → routing decision → possible LTFU → colposcopy → CIN grade → treatment/surveillance. Reading a trace answers questions like:
- Did every patient with HSIL attempt colposcopy?
- Did every patient with CIN2+ get routed to LEEP?
- Did patients who were lost to follow-up exit cleanly without any downstream steps?

We loop until 10 cervical-eligible patients have been traced — some sampled patients won't be eligible (too young, no cervix, etc.) and are skipped.

In [ ]:
random.seed(99)
DAY     = 0
metrics = initialize_metrics()

# We loop until we have 10 complete end-to-end traces.
# Not every sampled patient will be usable: some won't be cervical-eligible
# (wrong age or no cervix), and some will be skipped by run_screening_step()
# (not yet due for screening). We just keep sampling until 10 go through.
n_run, traced, pid = 0, [], 0

while n_run < 10:
    p = sample_patient(pid, DAY, 'gynecologist', 'outpatient')
    pid += 1  # increment id so each patient is unique

    # Skip patients not eligible for cervical screening
    if 'cervical' not in get_eligible_screenings(p):
        continue

    # run_screening_step() checks eligibility + interval, assigns the test,
    # draws the result, and stores it on the patient. Returns None if skipped.
    result = run_screening_step(p, 'cervical', DAY)
    if result is None:
        continue

    # run_cervical_followup() chains: routing → (colposcopy) → (treatment/surveillance)
    # Each step may exit early if the patient is lost to follow-up
    disposition = run_cervical_followup(p, DAY, metrics)
    p.log(DAY, f'DISPOSITION: {disposition}')  # stamp final outcome on the event log
    traced.append(p)
    n_run += 1

# Print each patient's full event log — every action taken is recorded by p.log()
for p in traced:
    p.print_history()

## Cervical Disposition Summary

Aggregate counts from the patient traces above, displayed in the same format as the full simulation summary. Because we ran patient traces manually (bypassing the runner), we backfill the metrics dict by hand before calling `print_summary`.

In the full simulation runner (`04_simulation_runner.ipynb`), these metrics are populated automatically as the simulation advances. Here we do it manually so we can use the same reporting infrastructure on a small hand-traced cohort — a useful pattern for debugging a specific pathway without running the full model.

In [ ]:
# In the full simulation runner, metrics are populated automatically as events fire.
# Here we bypassed the runner (to get individual traces), so we backfill the counts manually.
# This lets us call print_summary() and see the same aggregate view as the runner would produce.
metrics['n_patients']     = len(traced)
metrics['n_eligible_any'] = len(traced)

for p in traced:
    metrics['n_screened']['cervical'] += 1

    # Tally result categories (NORMAL, ASCUS, etc.) for the distribution breakdown
    if p.cervical_result:
        metrics['cervical_results'][p.cervical_result] += 1

    # record_exit() increments the right exit-reason bucket (lost_to_followup, untreated, etc.)
    if p.exit_reason:
        record_exit(metrics, p.exit_reason)

print_summary(metrics)

## Cervical CIN Grade Distribution by Trigger (Visualization)

The stacked bar chart below shows how the **mix of CIN grades shifts as the triggering result escalates in severity** — from ASCUS (mildest) on the left to HSIL (most severe) on the right.

This is one of the most clinically important charts in the model: it shows that the choice of colposcopy trigger has a direct impact on what CIN grades will be found and, therefore, how many LEEP procedures will be needed. A model that doesn't capture this gradient would misestimate both the procedure volume and the revenue impact of different screening protocols.

**Reading the chart**: each bar totals 100%. Green (NORMAL) and yellow (CIN1) segments represent patients who will be managed with surveillance. Orange (CIN2) and dark red (CIN3) segments represent patients who will receive LEEP excision.

In [ ]:
N = 2_000
triggers   = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
cin_grades = ['NORMAL', 'CIN1', 'CIN2', 'CIN3']
colors     = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']  # green → dark red (severity)

# For each trigger, run N colposcopies and record the fraction that landed on each CIN grade.
# fractions[trigger] becomes a list of 4 fractions (one per grade) that sum to 1.
fractions = {}
for trigger in triggers:
    counts = Counter()
    for _ in range(N):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, trigger
        counts[run_colposcopy(p, 0)] += 1
    fractions[trigger] = [counts.get(g, 0) / N for g in cin_grades]

# Build a stacked bar chart: each bar = one trigger, each colour segment = one CIN grade.
# We iterate over grades and call ax.bar() repeatedly, each time passing the previous
# bar's top as the 'bottom' parameter so grades stack on top of each other.
fig, ax = plt.subplots(figsize=(9, 5))
bottoms = [0] * len(triggers)
for i, grade in enumerate(cin_grades):
    vals = [fractions[t][i] * 100 for t in triggers]
    ax.bar(triggers, vals, bottom=bottoms, color=colors[i], label=grade)
    bottoms = [b + v for b, v in zip(bottoms, vals)]  # shift baseline up for the next grade

ax.set_ylabel('CIN grade distribution (%)')
ax.set_title('Colposcopy CIN grade mix by triggering result')
ax.legend(title='CIN grade', loc='upper right')
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

## Cervical LTFU Waterfall

Loss-to-follow-up is not a single event — it can happen at multiple points in the cervical pathway. A patient with an abnormal result might:
1. Never book a colposcopy appointment (**post-abnormal LTFU**).
2. Complete colposcopy but never follow through on treatment (**post-colposcopy LTFU**).
3. Complete treatment successfully.
4. Be placed on surveillance (CIN1/NORMAL → watchful waiting).

The waterfall below simulates 1,000 patients who all start with an HSIL result — the worst-case scenario — and tracks how many make it through each step. The drop from one bar to the next is directly attributable to LTFU at that node. These drop-off rates are controlled by `config.LTFU_PROBS` and should be calibrated against NYP's actual attrition data.

In [ ]:
random.seed(42)
N = 1_000

# Force every patient to start with HSIL — the most severe cytology result.
# All HSIL patients should attempt colposcopy; what happens after depends on
# two stochastic nodes: LTFU post-abnormal and LTFU post-colposcopy.
n_abnormal  = N
n_colposcopy, n_treatment_reached, n_treated, n_surveillance, n_ltfu = 0, 0, 0, 0, 0

for _ in range(N):
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix, p.cervical_result = 42, True, 'HSIL'
    m = initialize_metrics()  # fresh metrics dict per patient — we read it after

    # run_cervical_followup() returns a disposition string that tells us
    # exactly where the patient ended up in the pipeline
    disposition = run_cervical_followup(p, 0, m)

    # Classify how far this patient got based on the disposition and patient state.
    # Note: a patient can reach colposcopy but still exit afterward (LTFU post-colposcopy).
    if disposition != 'exit' or p.colposcopy_result:
        n_colposcopy += 1         # colposcopy was performed (result is on the patient)
    if disposition in ('treated', 'surveillance'):
        n_treatment_reached += 1  # made it to the treatment decision step
    if disposition == 'treated':
        n_treated += 1            # CIN2/3 → received LEEP excision
    if disposition == 'surveillance':
        n_surveillance += 1       # CIN1/normal → watchful waiting, no procedure
    if disposition == 'exit' or not p.active:
        n_ltfu += 1               # dropped out at some point in the pipeline

# Build the waterfall: each bar shows what % of the original 1,000 reached that stage
stages = ['Abnormal result', 'Reached colposcopy', 'Reached treatment step', 'Excisional tx', 'Surveillance']
counts = [n_abnormal, n_colposcopy, n_treatment_reached, n_treated, n_surveillance]
pcts   = [c / N * 100 for c in counts]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stages, pcts, color=['#1565C0', '#1976D2', '#42A5F5', '#EF5350', '#66BB6A'])
ax.set_ylabel('% of patients (starting from HSIL result)')
ax.set_title('Cervical follow-up waterfall — HSIL cohort (n=1,000)')
ax.set_ylim(0, 115)
# Annotate each bar with both the percentage and the absolute patient count
for bar, pct, cnt in zip(bars, pcts, counts):
    ax.text(bar.get_x() + bar.get_width()/2, pct + 1.5,
            f'{pct:.1f}%\n(n={cnt})', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## Lung Follow-Up Routing

Lung follow-up is structured differently from cervical: the Lung-RADS category after LDCT determines not just *whether* follow-up is needed, but *what kind* and *how soon*. The clinical flow is:

| RADS category | Finding | Next step |
|---|---|---|
| RADS 0 | Incomplete scan | Repeat LDCT in 1–3 months |
| RADS 1/2 | Negative / benign | Routine repeat LDCT in 12 months |
| RADS 3 | Probably benign | Repeat LDCT in 6 months (nodule surveillance) |
| RADS 4A/4B/4X | Suspicious | Biopsy pathway (5 sequential LTFU nodes) |

The biopsy pathway for RADS 4 patients is the most complex: result must be communicated → biopsy referral placed → biopsy scheduled → biopsy completed → pathology confirms malignancy → treatment given. Each step has its own drop-out probability, and any failure terminates the patient as LTFU.

In [ ]:
# All Lung-RADS categories in config order (RADS_0 through RADS_4B_4X)
rads_categories = list(cfg.LUNG_RADS_PROBS.keys())

# 500 trials per category because:
# - RADS 0/1/2/3: the only stochastic node is result communication (~10% LTFU),
#   so the routing is mostly deterministic once that passes
# - RADS 4A/4B/4X: there are 5 sequential stochastic nodes (referral, scheduling,
#   completion, malignancy, treatment), producing a wider range of dispositions
N_TRIALS = 500

print(f'{"RADS category":<15}  Routing distribution ({N_TRIALS} trials each)')
print('-' * 75)
for rads in rads_categories:
    dispositions = []
    for _ in range(N_TRIALS):
        # Typical eligible lung patient: 62-year-old active smoker with 30 pack-years
        p = sample_patient(0, 0, 'specialist', 'outpatient')
        p.age, p.smoker, p.pack_years = 62, True, 30
        p.lung_result = rads  # set the result we want to follow up on

        # run_lung_followup() reads p.lung_result, applies the RADS routing logic,
        # and returns a string describing the final disposition
        dispositions.append(run_lung_followup(p, 0))

    # Count how many trials ended in each disposition
    cnt     = Counter(dispositions)
    summary = '  '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {rads:<13}  {summary}')

## Lung Biopsy Pathway — LTFU Funnel

This funnel shows the cumulative attrition across the biopsy pathway for RADS 4A patients — the most common category requiring a biopsy.

Each bar is shorter than the one above because some fraction of patients was lost at the previous step. The goal of a coordinated care program is to make these bars as equal in height as possible — i.e. to close the gap between referral and completion at every step.

The funnel is generated from 2,000 simulated RADS 4A patients to get stable estimates of each step's drop-off rate. Each patient's metrics dict is checked after the run to see which steps were actually completed.

In [ ]:
random.seed(42)
N = 2_000  # RADS_4A — most common biopsy-pathway category

step_counts = defaultdict(int)

for _ in range(N):
    p = sample_patient(0, 0, 'specialist', 'outpatient')
    p.age, p.smoker, p.pack_years = 62, True, 30
    p.lung_result = 'RADS_4A'
    m = initialize_metrics()   # fresh metrics dict so we can read per-patient step flags
    run_lung_followup(p, 0, m)

    # run_lung_followup() increments a metrics key at each step it successfully completes.
    # If a step is never reached (patient was lost earlier), that key stays at 0.
    # Summing m.get(key, 0) across all N patients gives the cohort count for each node.
    step_counts['Result communicated']  += m.get('lung_result_communicated', 0)
    step_counts['Biopsy referral made'] += m.get('lung_biopsy_referral', 0)
    step_counts['Biopsy scheduled']     += m.get('lung_biopsy_scheduled', 0)
    step_counts['Biopsy completed']     += m.get('lung_biopsy_completed', 0)
    step_counts['Malignancy confirmed'] += m.get('lung_malignancy_confirmed', 0)
    step_counts['Treatment given']      += m.get('lung_treatment_given', 0)

# Convert raw counts to % of the starting cohort so the bars are comparable
steps  = ['Result communicated', 'Biopsy referral made', 'Biopsy scheduled',
          'Biopsy completed', 'Malignancy confirmed', 'Treatment given']
counts = [step_counts[s] for s in steps]
pcts   = [c / N * 100 for c in counts]

# Horizontal funnel: we reverse the lists so the first step appears at the top.
# Each bar is shorter than the one above it, showing how the cohort shrinks.
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(steps[::-1], pcts[::-1],
               color=['#1565C0','#1976D2','#42A5F5','#64B5F6','#EF5350','#66BB6A'][::-1])
ax.set_xlabel('% of RADS 4A patients (n=2,000)')
ax.set_title('Lung biopsy pathway — LTFU funnel (RADS 4A cohort)')
ax.set_xlim(0, 110)
# Annotate each bar with its % and the raw patient count
for bar, pct, cnt in zip(bars, pcts[::-1], counts[::-1]):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%  (n={cnt})', va='center', fontsize=9)
plt.tight_layout()
plt.show()